# Independent Component Analysis (ICA)

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/independent-component-analysis)

Implements the cocktail party problem with ICA: mixing independent audio-like sources, then recovering them using the Bell-Sejnowski algorithm and FastICA. Visualizes how non-Gaussianity is the key to unmixing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d27',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#94a3b8',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'axes.edgecolor': '#2d3748',
    'grid.color': '#2d3748',
    'axes.grid': True,
})

np.random.seed(0)

## 1. The cocktail party problem

Three independent sources (sine, sawtooth, uniform noise) mixed by an unknown matrix A.

In [ ]:
n = 2000
t = np.linspace(0, 8, n)

# Independent, non-Gaussian sources
s1 = np.sin(2 * np.pi * 1.3 * t)                    # sine wave (sub-Gaussian)
s2 = 2 * (t * 1.7 % 1) - 1                          # sawtooth (uniform-like)
s3 = np.random.uniform(-1, 1, n)                    # pure uniform noise

S = np.column_stack([s1, s2, s3])  # (n, 3) — sources
S_std = (S - S.mean(0)) / S.std(0)  # standardize

# Random mixing matrix
A = np.array([[1.0, 0.5, 0.3],
              [0.4, 1.0, 0.6],
              [0.7, 0.2, 1.0]])
X_mixed = S_std @ A.T  # (n, 3) — observed mixtures

fig, axes = plt.subplots(3, 2, figsize=(14, 6), sharex=True)
source_names = ['Sine', 'Sawtooth', 'Uniform noise']
colors_s = ['#6366f1', '#10b981', '#f59e0b']

for i in range(3):
    axes[i, 0].plot(t[:500], S_std[:500, i], color=colors_s[i], linewidth=1)
    axes[i, 0].set_ylabel(source_names[i], color='#94a3b8')
    axes[i, 1].plot(t[:500], X_mixed[:500, i], color='#94a3b8', linewidth=1)
    axes[i, 1].set_ylabel(f'Mix {i+1}', color='#94a3b8')

axes[0, 0].set_title('Original sources', color='#e2e8f0')
axes[0, 1].set_title('Observed mixtures (what microphones hear)', color='#e2e8f0')
for ax in axes[-1]:
    ax.set_xlabel('Time')
plt.tight_layout()
plt.show()

## 2. Why non-Gaussianity is the key

The Central Limit Theorem tells us that *mixtures are more Gaussian than sources*. So ICA finds the unmixing that maximizes non-Gaussianity.

In [ ]:
def kurtosis(x):
    """Excess kurtosis (0 for Gaussian, >0 for heavy-tailed, <0 for light-tailed)."""
    x = x - x.mean()
    return np.mean(x**4) / (np.mean(x**2)**2) - 3

print("Kurtosis of sources (should be non-zero):")
for i, name in enumerate(source_names):
    print(f"  {name}: {kurtosis(S_std[:, i]):.3f}")

print("\nKurtosis of mixtures (closer to 0 = more Gaussian after mixing):")
for i in range(3):
    print(f"  Mix {i+1}: {kurtosis(X_mixed[:, i]):.3f}")

# Histograms showing Gaussianization
fig, axes = plt.subplots(2, 3, figsize=(12, 5))
gauss_x = np.linspace(-3, 3, 100)
from scipy.stats import norm as sp_norm
for i in range(3):
    axes[0, i].hist(S_std[:, i], bins=60, density=True, color=colors_s[i], alpha=0.8)
    axes[0, i].plot(gauss_x, sp_norm.pdf(gauss_x), 'w--', linewidth=1.5, alpha=0.6)
    axes[0, i].set_title(f'{source_names[i]}\nkurt={kurtosis(S_std[:,i]):.2f}', color='#e2e8f0')
    axes[1, i].hist(X_mixed[:, i], bins=60, density=True, color='#94a3b8', alpha=0.8)
    axes[1, i].plot(gauss_x, sp_norm.pdf(gauss_x), 'w--', linewidth=1.5, alpha=0.6)
    axes[1, i].set_title(f'Mix {i+1}\nkurt={kurtosis(X_mixed[:,i]):.2f}', color='#e2e8f0')

axes[0, 0].set_ylabel('Sources', color='#94a3b8')
axes[1, 0].set_ylabel('Mixtures', color='#94a3b8')
plt.suptitle('Mixing Makes Distributions More Gaussian (CLT)', color='#e2e8f0')
plt.tight_layout()
plt.show()

## 3. FastICA — recover the sources

In [ ]:
def fast_ica(X, n_components, max_iter=500, tol=1e-6):
    """
    FastICA using negentropy approximation G(u) = log cosh(u).
    Returns recovered sources S_hat of shape (n, n_components).
    """
    # Whiten: decorrelate and scale to unit variance
    X_c = X - X.mean(0)
    cov = X_c.T @ X_c / len(X)
    eigvals, eigvecs = np.linalg.eigh(cov)
    # Sort descending
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
    W_white = eigvecs / np.sqrt(eigvals) @ eigvecs.T  # whitening matrix
    X_white = X_c @ W_white.T  # (n, p)

    p = X_white.shape[1]
    W = np.random.randn(n_components, p)
    # Orthonormalize
    W, _, _ = np.linalg.svd(W, full_matrices=False)

    for it in range(max_iter):
        W_old = W.copy()
        # G(u) = log cosh(u), g(u) = tanh(u), g'(u) = 1 - tanh²(u)
        u = X_white @ W.T  # (n, k)
        g_u = np.tanh(u)           # g applied to u
        g_prime_u = 1 - g_u**2     # g'
        W_new = (g_u.T @ X_white) / len(X) - g_prime_u.mean(0)[:, None] * W
        # Symmetric orthogonalization
        U, S_s, Vt = np.linalg.svd(W_new)
        W = U @ Vt
        # Check convergence
        delta = np.max(np.abs(np.abs((W * W_old).sum(1)) - 1))
        if delta < tol:
            break

    S_hat = X_white @ W.T
    return S_hat, W

S_hat, W_ica = fast_ica(X_mixed, n_components=3)

# Match and sign-flip recovered sources to originals (for display)
def match_sources(S_true, S_hat):
    corr = np.abs(np.corrcoef(S_true.T, S_hat.T)[:3, 3:])
    order = corr.argmax(1)
    signs = [np.sign(np.corrcoef(S_true[:, i], S_hat[:, order[i]])[0, 1]) for i in range(3)]
    return S_hat[:, order] * signs

S_recovered = match_sources(S_std, S_hat)

fig, axes = plt.subplots(3, 2, figsize=(14, 6), sharex=True)
for i in range(3):
    axes[i, 0].plot(t[:500], S_std[:500, i], color=colors_s[i], linewidth=1)
    axes[i, 0].set_ylabel(f'True {source_names[i]}', color='#94a3b8')
    axes[i, 1].plot(t[:500], S_recovered[:500, i], color=colors_s[i], linewidth=1, alpha=0.8)
    corr = np.corrcoef(S_std[:, i], S_recovered[:, i])[0, 1]
    axes[i, 1].set_ylabel(f'Recovered (r={corr:.3f})', color='#94a3b8')

axes[0, 0].set_title('True sources', color='#e2e8f0')
axes[0, 1].set_title('ICA-recovered sources', color='#e2e8f0')
for ax in axes[-1]: ax.set_xlabel('Time')
plt.suptitle('FastICA Successfully Unmixes the Sources', color='#e2e8f0')
plt.tight_layout()
plt.show()

## ✏️ Your turn

**Exercise 1 — ICA fails for Gaussian sources.** Generate 3 independent Gaussian sources (instead of sine/sawtooth/uniform), mix them, and apply FastICA. Show that the recovered sources do NOT match the originals. Explain why in a comment.

In [ ]:
# TODO(you): create 3 Gaussian sources, mix them, run FastICA, measure correlations
# S_gauss = np.random.randn(n, 3)
# X_gauss_mixed = S_gauss @ A.T
# S_gauss_hat, _ = fast_ica(X_gauss_mixed, 3)
# Measure correlations — they should be low / random

In [ ]:
# Assert cell
S_gauss = np.random.randn(n, 3)
X_gauss_mixed = S_gauss @ A.T
S_gauss_hat, _ = fast_ica(X_gauss_mixed, 3)
corrs = [np.max(np.abs(np.corrcoef(S_gauss[:, i], S_gauss_hat.T)[0, 1:])) for i in range(3)]
print("Max correlation per source (Gaussian case):", [f"{c:.3f}" for c in corrs])
print("Compare to non-Gaussian case where correlations should be > 0.95")
# For Gaussian sources ICA typically gives low alignment (< 0.8)
print("ICA on Gaussian sources cannot reliably recover independent components.")

<details><summary>Solution</summary>

```python
S_gauss = np.random.randn(n, 3)
X_gauss_mixed = S_gauss @ A.T
S_gauss_hat, _ = fast_ica(X_gauss_mixed, 3)
S_gauss_rec = match_sources(S_gauss, S_gauss_hat)

corrs = [np.corrcoef(S_gauss[:, i], S_gauss_rec[:, i])[0, 1] for i in range(3)]
print("Correlations (Gaussian sources):", [f"{c:.3f}" for c in corrs])
# These will be low / inconsistent
```

**Why ICA fails:** For Gaussian distributions, any rotation of independent Gaussians produces another set of independent Gaussians — so there is no unique unmixing matrix. The kurtosis of a Gaussian is exactly 0, so maximizing non-Gaussianity gives no gradient signal. In other words, the CLT argument reverses: Gaussian sources are *already at the maximum-entropy fixed point*, so ICA can't distinguish the mixed from the unmixed version.

</details>